# EpiCoV 2020

- Number of sequences: `200,524`
- Collection dates: 1 Jan 20 to 6 Jul 20

In [1]:
# General
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Custom helpers
from utils import (standardize_country_name, get_continent, choropleth_continent,
choropleth_world, map_age_to_group, col_fillna, plot_dominant_choropleth, inclusion_exclusion,
standardize_vaccine_status, map_pat_status)

import warnings
warnings.filterwarnings("ignore")

In [2]:
YEAR = "2020"
DATA_PATH = os.path.expanduser(f"~/gcs-data/Full_data/epicov_{YEAR}_200k.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.expanduser(f"~/gcs-data/Full_data/epicov_{YEAR}_200k.parquet")
PLOTS_PATH = f"Nov25_plots/epicov_{YEAR}/"

os.makedirs(PLOTS_PATH, exist_ok=True)

The patient status is varying too much, let's standardize it using the manual annotaion by Dr. Miae Lee.

In [3]:
# For patient status mapping
df_pat = pd.read_excel(os.path.expanduser(f"~/gcs-data/patient_status_mapping_MLedit.xlsx"))
df_pat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118 entries, 0 to 117
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   pat_stat             117 non-null    object
 1   clinical status      118 non-null    object
 2   hospitalized status  118 non-null    object
 3   severity             118 non-null    object
 4   category             118 non-null    object
 5   remarks              49 non-null     object
 6   further remarks      35 non-null     object
dtypes: object(7)
memory usage: 6.6+ KB


In [4]:
%%time
df = pd.read_csv(DATA_PATH) if DATA_PATH.endswith(".csv") else pd.read_parquet(DATA_PATH)
df.sort_values(by=["Collection date"], inplace=True)
df.shape, df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200524 entries, 8434 to 200523
Data columns (total 18 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   Accession ID                     200524 non-null  object
 1   Submission date                  200524 non-null  object
 2   Virus name                       200524 non-null  object
 3   Collection date                  200524 non-null  object
 4   Location                         200524 non-null  object
 5   Host                             200524 non-null  object
 6   Additional location information  6210 non-null    object
 7   Sampling strategy                11252 non-null   object
 8   Gender                           200524 non-null  object
 9   Patient age                      200519 non-null  object
 10  Patient status                   200509 non-null  object
 11  Last vaccinated                  76 non-null      object
 12  Passage           

((200524, 18), None)

In [5]:
df.nunique()

Accession ID                       200524
Submission date                      1016
Virus name                         200478
Collection date                       188
Location                             4418
Host                                    1
Additional location information      1035
Sampling strategy                     111
Gender                                133
Patient age                           311
Patient status                        133
Last vaccinated                        12
Passage                               103
Specimen                              329
Additional host information           417
Lineage                               946
Clade                                  11
AA Substitutions                    70260
dtype: int64

## Clean the data

Let's fill in missing values, fix spurious values and standardize a few column values

### Gender

In [6]:
df["Gender"].value_counts()

Gender
unknown    126740
Male        38933
Female      34391
49.0           17
unknowm        15
            ...  
64              1
9               1
13              1
62.0            1
23.0            1
Name: count, Length: 133, dtype: int64

In [7]:
a = "100"
a[0].isdigit()

True

In [8]:
[x for x in df["Patient status"].unique() if str(x)[0].isdigit()], [x for x in df["Gender"].unique() if str(x)[0].isdigit()]

([],
 ['3.0',
  '66.0',
  '57.0',
  '69.0',
  '76.0',
  '38.0',
  '63.0',
  '65',
  '58.0',
  '41.0',
  '37.0',
  '34.0',
  '64.0',
  '49.0',
  '31.0',
  '46.0',
  '17.0',
  '39.0',
  '47.0',
  '74.0',
  '42.0',
  '48.0',
  '35.0',
  '79.0',
  '65.0',
  '60.0',
  '33.0',
  '32.0',
  '24.0',
  '68.0',
  '75.0',
  '29.0',
  '52.0',
  '73.0',
  '40.0',
  '26.0',
  '70.0',
  '53.0',
  '22.0',
  '45.0',
  '50.0',
  '92.0',
  '25.0',
  '27.0',
  '21.0',
  '59.0',
  '16.0',
  '41',
  '59',
  '47',
  '46',
  '30',
  '53',
  '25',
  '20',
  '18.0',
  '12',
  '33',
  '55',
  '28.0',
  '36.0',
  '44',
  '20.0',
  '51.0',
  '19.0',
  '51',
  '57',
  '43',
  '52',
  '61',
  '45',
  '27',
  '28',
  '24',
  '35',
  '39',
  '22',
  '32',
  '49',
  '61.0',
  '34',
  '26',
  '75',
  '38',
  '48',
  '58',
  '16',
  '42',
  '37',
  '29',
  '21',
  '72',
  '56',
  '62',
  '40',
  '31',
  '74',
  '14',
  '50',
  '36',
  '63',
  '84.0',
  '54',
  '73',
  '54.0',
  '18',
  '23',
  '43.0',
  '6',
  '17',
  '67

In [9]:
genders = ["male", "unknown", "female"]

indices = df.index[df["Gender"].isin([x for x in df["Gender"].unique() if str(x)[0].isdigit()])].tolist()
print(len(indices))
for i in indices:
    age = df.at[i, "Patient age"]
    if str(age).lower() in genders:
        df.at[i, "Patient age"] = df.at[i, "Gender"]
        df.at[i, "Gender"] = age
df["Gender"].value_counts()

445


Gender
unknown    126872
Male        39090
Female      34546
unknowm        15
0               1
Name: count, dtype: int64

Still one entry is left. Let's see what's the issue.

In [10]:
df[df["Gender"]=="0"]

,Accession ID,Submission date,Virus name,Collection date,Location,Host,Additional location information,Sampling strategy,Gender,Patient age,Patient status,Last vaccinated,Passage,Specimen,Additional host information,Lineage,Clade,AA Substitutions
192363,EPI_ISL_7147154,2021-12-04,hCoV-19/USA/WY-WYPHL-20033054/2020,2020-06-26,North America / USA / Wyoming,Human,NaN,NaN,0,18,unknown,NaN,Original,NaN,NaN,B.1.166,GH,"(Spike_D574Y,NS3_Q57H,NSP12_P323L,Spike_D614G,..."


In [11]:
indices = df.index[df["Gender"].isin(["unknowm", "0"])].tolist()
df.loc[indices, "Gender"] = "unknown"
df["Gender"].value_counts()

Gender
unknown    126888
Male        39090
Female      34546
Name: count, dtype: int64

### Age

In [12]:
import re
from collections import defaultdict

def find_age_formats(df):
    # Identify regex patterns for different Patient age formats and show examples/counts
    ages = pd.Series(df["Patient age"].astype(str).fillna("").values).str.strip()
    unique_ages = pd.Series(ages.unique())

    patterns = {
        "integer_years": r'^\s*\d{1,3}\s*(?:y|yr|yrs|year|years)?\s*$',
        "decimal_years": r'^\s*\d+\.\d+\s*(?:y|yr|yrs|year|years)?\s*$',
        "months": r'^\s*\d{1,3}\s*(?:m|mo|mos|month|months)\b\.?$',
        "days": r'^\s*\d{1,3}\s*(?:d|day|days)\b\.?$',
        "age_range": r'^\s*(?:<\s*)?\d{1,3}\s*(?:-|–|to)\s*\d{1,3}\s*(?:y|yr|yrs|year|years)?\s*$',
        "decade_s": r'^\s*\d{2}s\s*$',
        "less_than": r'^\s*[<>]\s*\d{1,3}\s*(?:y|yr|yrs|year|years)?\s*$',
        "birth_year": r'^\s*(?:19|20)\d{2}\s*$',
        "text_labels": r'^\s*(?:newborn|neonate|infant|baby|child|teen(?:ager)?|adolescent|adult|elderly|senior|unknown|unk|n/?a|na|not available)\b',
    }

    compiled = {k: re.compile(v, re.I) for k, v in patterns.items()}

    matches = defaultdict(list)
    for val in unique_ages.dropna().astype(str):
        v = val.strip()
        matched = False
        for name, cre in compiled.items():
            if v and cre.search(v):
                matches[name].append(v)
                matched = True
        if not matched and v:
            matches["unmatched"].append(v)

    # Print summary: count and up to 10 examples for each pattern
    for name in list(compiled.keys()) + ["unmatched"]:
        vals = pd.Series(matches.get(name, [])).drop_duplicates()
        print(f"{name}: {len(vals)} unique matches")
        if len(vals) > 0:
            print(vals.head(10).tolist())
        print("-" * 60)

    # Optional: show top unmatched values to refine regexes
    if matches.get("unmatched"):
        unmatched_counts = ages[ages.isin(matches["unmatched"])].value_counts().head(30)
        print("Top unmatched values (sample counts):")
        print(unmatched_counts)
    
    return matches

In [13]:
matches = find_age_formats(df)

integer_years: 122 unique matches
['73', '56', '72', '32', '35', '27', '50', '45', '36', '30']
------------------------------------------------------------
decimal_years: 72 unique matches
['3.0', '27.0', '19.0', '66.0', '57.0', '69.0', '48.0', '59.0', '0.33', '76.0']
------------------------------------------------------------
months: 19 unique matches
['3 months', '6 months', '4 months', '10 months', '2 months', '7 months', '9 months', '11 months', '5 months', '1 month']
------------------------------------------------------------
days: 4 unique matches
['8 days', '10 days', '21 days', '27 days']
------------------------------------------------------------
age_range: 98 unique matches
['55-59', '30-40', '40-50', '60-70', '50-54', '60-64', '70-74', '80-84', '65-69', '31 - 40']
------------------------------------------------------------
decade_s: 6 unique matches
['70s', '80s', '90s', '50s', '20s', '60s']
------------------------------------------------------------
less_than: 12 uniqu

From the above analysis, we see that its either of the following formats:
- `Integer years`: Convert to `float` for consistency
- `Decimal years`: Convert to `float` for consistency
- `Months`: Divide by `12` to get the age in years (1 decimal places)
- `Days`: Since all are less than `30 days` we assign as `0.0` (age in years)
- `Age range`: Assign the mean age (in `age_range` in column, we retain age range and handle standarization)
- `Decade`: Assign the mean age (in `age_range` in column, we retain age range and handle standarization)
- `Greater than`: Assign the lower bound (in `age_range` in column, we retain age range and handle standarization)
- `Text`: They are just different cases of `unknown` so standardize to lower case
- `Unmatched formats`:
    - `nan`, `unkown`, `unknow`: Map to `unknown`
    - Others seem to be like a subtraction operation ==> perform subtraction and have the float value

In [14]:
df[df["Patient age"].isin(["Male", "Female"])]

,Accession ID,Submission date,Virus name,Collection date,Location,Host,Additional location information,Sampling strategy,Gender,Patient age,Patient status,Last vaccinated,Passage,Specimen,Additional host information,Lineage,Clade,AA Substitutions
153446,EPI_ISL_4404950,2021-09-22,hCoV-19/Argentina/PAIS-A0768/2020,2020-06-16,South America / Argentina / Buenos Aires / Ber...,Human,NaN,NaN,unknown,Male,unknown,NaN,Original,NaN,NaN,B.1.499,GH,"(NSP7_S25L,NS3_Q57H,NSP2_T85I,NSP14_A320V,N_S1..."
191444,EPI_ISL_4404965,2021-09-22,hCoV-19/Argentina/PAIS-A0778/2020,2020-06-22,South America / Argentina / Buenos Aires / Ber...,Human,NaN,NaN,unknown,Male,unknown,NaN,Original,NaN,NaN,B.1.499,GH,"(NSP7_S25L,NS3_Q57H,NSP2_T85I,NSP15_S147I,NSP1..."
190334,EPI_ISL_4220000,2021-09-15,hCoV-19/Mexico/CMX-INMEGEN-04-07-79/2020,2020-06-24,North America / Mexico / Mexico City,Human,NaN,NaN,unknown,Female,unknown,NaN,Original,Oropharyngeal swab,NaN,B.1.609,G,"(N_S37P,NSP12_P323L,Spike_D614G,NSP12_Q191R)"
192382,EPI_ISL_4219998,2021-09-15,hCoV-19/Mexico/CMX-INMEGEN-04-07-77/2020,2020-06-24,North America / Mexico / Mexico City,Human,NaN,NaN,unknown,Female,unknown,NaN,Original,Oropharyngeal swab,NaN,B.1.1.222,GR,"(Spike_T732A,N_R203K,N_G204R,NSP15_V172L,NSP13..."
192381,EPI_ISL_4219999,2021-09-15,hCoV-19/Mexico/CMX-INMEGEN-04-07-78/2020,2020-06-24,North America / Mexico / Mexico City,Human,NaN,NaN,unknown,Female,unknown,NaN,Original,Oropharyngeal swab,NaN,B.1.369,GH,"(N_S183Y,NS3_Q57H,NSP2_T85I,NSP12_P323L,Spike_..."
143812,EPI_ISL_4220060,2021-09-15,hCoV-19/Mexico/CMX-INMEGEN-04-07-181/2020,2020-07-02,North America / Mexico / Mexico City,Human,NaN,NaN,unknown,Male,unknown,NaN,Original,Oropharyngeal swab,NaN,B.1.189,G,"(NSP15_D128Y,NSP3_C55Y,NSP3_V1229F,NSP12_A449V..."


In [15]:
for i,row in df[df["Patient age"].isin(["Male", "Female"])].iterrows():
    if row["Gender"].lower() in ["nan", "unknown"]:
        df.at[i, "Gender"] = row["Patient age"]
        df.at[i, "Patient age"] = "unknown"

In [16]:
matches1 = find_age_formats(df)

integer_years: 122 unique matches
['73', '56', '72', '32', '35', '27', '50', '45', '36', '30']
------------------------------------------------------------
decimal_years: 72 unique matches
['3.0', '27.0', '19.0', '66.0', '57.0', '69.0', '48.0', '59.0', '0.33', '76.0']
------------------------------------------------------------
months: 19 unique matches
['3 months', '6 months', '4 months', '10 months', '2 months', '7 months', '9 months', '11 months', '5 months', '1 month']
------------------------------------------------------------
days: 4 unique matches
['8 days', '10 days', '21 days', '27 days']
------------------------------------------------------------
age_range: 98 unique matches
['55-59', '30-40', '40-50', '60-70', '50-54', '60-64', '70-74', '80-84', '65-69', '31 - 40']
------------------------------------------------------------
decade_s: 6 unique matches
['70s', '80s', '90s', '50s', '20s', '60s']
------------------------------------------------------------
less_than: 12 uniqu

In [17]:
def multi_unit_age(age:str):
    """
    Convert age strings with multiple units (e.g., "1 month 14 days") to age in years (float).
    Assumes 1 year = 12 months, 1 month = 30 days for conversion.
    """
    # Use the combined pattern with OR logic
    pattern = re.compile(
        # 1. Matches 'X years Y months' (looks for 'y' then 'm')
        r'^\s*(?P<years>\d+)\s*y\w*?\s*(?:[,;\-]?\s*)?(?P<val_months>\d+)\s*m\w*?\s*$'
        
        + '|' # OR 
        
        # 2. Matches 'X months Y days' (looks for 'm' then 'd')
        r'^\s*(?P<months>\d+)\s*m\w*?\s*(?:[,;\-]?\s*)?(?P<days>\d+)\s*d\w*?\s*$',
        re.I 
    )

    def calculate_total_years(duration_string):
        match = pattern.search(duration_string)
        
        if not match:
            return -1

        # Use a dictionary to easily check which group was captured (since OR groups return None for the unmatched side)
        data = match.groupdict()
        
        total_years = 0.0

        # --- Case 1: Years/Months Structure Matched ---
        if data['years'] is not None:
            # data['years'] is the first number (X), data['val_months'] is the second (Y)
            Y = int(data['years'])
            M = int(data['val_months'])
            
            # Calculation: Y + M/12
            total_years = Y + (M / 12.0)
            unit = "Years/Months"

        # --- Case 2: Months/Days Structure Matched ---
        elif data['months'] is not None:
            # data['months'] is the first number (X), data['days'] is the second (Y)
            M = int(data['months'])
            D = int(data['days'])
            
            # Calculation: M/12 + D/365.25 (using 365.25 days/year for accuracy)
            total_years = (M / 12.0) + (D / 365.25)
            unit = "Months/Days"
            
        else:
            # Should not happen if one of the OR conditions is met
            return f"Error: Failed to extract values from '{duration_string}'"

        return round(total_years, 1)
    return calculate_total_years(age)

In [20]:
matches = matches1
for i,row in df.iterrows():
    age = str(row["Patient age"]).strip()
    if age in matches["integer_years"] or age in matches["decimal_years"]:
        df.at[i, "Patient age"] = float(re.findall(r'\d+\.?\d*', age)[0])
    elif age in matches["text_labels"]:
        df.at[i, "Patient age"] = "unknown"
    elif age in matches["months"]:
        months = float(re.findall(r'\d+\.?\d*', age)[0])
        df.at[i, "Patient age"] = round(months / 12, 1)
    elif age in matches["days"]:
        days = float(re.findall(r'\d+\.?\d*', age)[0])
        df.at[i, "Patient age"] = 0.0
    elif age in matches["decade_s"]:
        decade = int(re.findall(r'\d{2}', age)[0])
        df.at[i, "Patient age"] = f"{decade}-{decade+9}"
    elif age in matches["less_than"]:
        val = float(age[1:])
        if age.startswith("<"):
            df.at[i, "Patient age"] = val - 5  # Assuming '< X' means X - 5 years
        elif age.startswith(">"):
            df.at[i, "Patient age"] = val + 5  # Assuming '> X' means X + 5 years
    elif age in matches["unmatched"]:
        if age.lower().startswith("u") or age == "nan" or age == 'X':
            df.at[i, "Patient age"] = "unknown"
        elif "-" in age:
            a,b = age.split("-") if "-" in age else age.split("to")
            if a == "2020":
                df.at[i, "Patient age"] = (float(a.strip()) - float(b.strip()))
            else:
                df.at[i, "Patient age"] = (float(a.strip()) + float(b.strip())) / 2
        elif "weeks" in age:
            weeks = float(re.findall(r'\d+\.?\d*', age)[0])
            df.at[i, "Patient age"] = round((weeks * 7) / 365, 1)
        elif "over" in age:
            val = float(re.findall(r'\d+\.?\d*', age)[0])
            print(age, val)
            df.at[i, "Patient age"] = val + 5  # Assuming 'over X' means X + 5 years
        elif "under" in age:
            val = float(re.findall(r'\d+\.?\d*', age)[0])
            print(age, val)
            df.at[i, "Patient age"] = val - 5  # Assuming 'under X' means X - 5 years
        elif multi_unit_age(age) != -1:
            val = multi_unit_age(age)
            print(age, val, "Success")
            df.at[i, "Patient age"] = val
        elif multi_unit_age(age) == -1 and re.match(r'^\s*\d{1,3}\s*+', age): # e.g. "90+"
            val = float(re.findall(r'\d+\.?\d*', age)[0])
            print(age, val)
            df.at[i, "Patient age"] = val + 5  # Assuming 'X+' means X + 5 years
    else:
        continue

over 100 100.0
1 month 14 days 0.1 Success
over 100 100.0
17 years 9 months 17.8 Success
90+ 90.0
5 years 4 months 5.3 Success
12 years 6 moths 12.5 Success
90+ 90.0
2 years 11 months 2.9 Success
4 years 6 moths 4.5 Success


In [21]:
find_age_formats(df)

integer_years: 0 unique matches
------------------------------------------------------------
decimal_years: 131 unique matches
['73.0', '56.0', '72.0', '32.0', '35.0', '27.0', '50.0', '45.0', '36.0', '30.0']
------------------------------------------------------------
months: 0 unique matches
------------------------------------------------------------
days: 0 unique matches
------------------------------------------------------------
age_range: 98 unique matches
['55-59', '30-40', '40-50', '60-70', '50-54', '60-64', '70-74', '80-84', '65-69', '31 - 40']
------------------------------------------------------------
decade_s: 0 unique matches
------------------------------------------------------------
less_than: 0 unique matches
------------------------------------------------------------
birth_year: 0 unique matches
------------------------------------------------------------
text_labels: 1 unique matches
['unknown']
------------------------------------------------------------
unmatche

defaultdict(list,
            {'text_labels': ['unknown'],
             'decimal_years': ['73.0',
              '56.0',
              '72.0',
              '32.0',
              '35.0',
              '27.0',
              '50.0',
              '45.0',
              '36.0',
              '30.0',
              '62.0',
              '64.0',
              '39.0',
              '37.0',
              '47.0',
              '44.0',
              '46.0',
              '21.0',
              '61.0',
              '3.0',
              '57.0',
              '68.0',
              '41.0',
              '83.0',
              '29.0',
              '66.0',
              '38.0',
              '15.0',
              '49.0',
              '65.0',
              '25.0',
              '88.0',
              '34.0',
              '52.0',
              '58.0',
              '40.0',
              '24.0',
              '85.0',
              '43.0',
              '67.0',
              '33.0',
              '74.0',
 

Let's retain the following columns:
- `Accession ID`
- `Collection date`
- `Submission date`
- `Location`
- `Additional location information`
- `Gender`
- `Patient age`
- `Patient status`
- `Last vaccinated`
- `Additional host information`

In [ ]:
cols = ["Accession ID", "Collection date", "Submission date", "Location", "Gender", "Patient status", "Patient age", "Additional location information", "Last vaccinated", "Additional host information"]
df = df[cols]
df.info()

In [ ]:
df["Last vaccinated"].value_counts()

In [ ]:
df[df["Last vaccinated"] == "Suspeito de reinfecção"]

`Suspeito de reinfecção` is Portugese for Suspected Re-infection. So, it can be considered as `Patient status` rather than vaccination status.

In [ ]:
index = df[df["Last vaccinated"] == "Suspeito de reinfecção"].index
df.loc[index, "Patient status"] = "Suspected Re-infection"

Next, let's standardize vaccination status

In [ ]:
df = standardize_vaccine_status(df)
df.shape, df["Last vaccinated"].value_counts()

According to [Ramarao-Milne et. al, 2022](https://www.csbj.org/article/S2001-0370(22)00219-7/fulltext), there is only `0.3%` of meaningful patient data in current available EpiCoV database. Let's see if this applies to our sample as well.

In [ ]:
inclusion_exclusion(df)

Next, let's standardize the patient status

In [ ]:
df["Patient status"] = df["Patient status"].fillna("Unknown")
df.info()

In [ ]:
df["Clinical status"], df["Hospitalization status"], df["Severity"], df["WHO category"] = np.nan, np.nan, np.nan, np.nan
df = map_pat_status(df, df_pat, "Clinical status")
df = map_pat_status(df, df_pat, "Hospitalization status")
df = map_pat_status(df, df_pat, "Severity")
df = map_pat_status(df, df_pat, "WHO category")

In [ ]:
df.info()

## Collection Submission delay statistics

In [ ]:
df["Submission date"] = pd.to_datetime(df["Submission date"], format="mixed")
df["Collection date"] = pd.to_datetime(df["Collection date"], format="mixed")

df["Collection date"].min(), df["Collection date"].max(), df["Submission date"].min(), df["Submission date"].max()

Let us validate if `Submission_date` is after the `Collection_date`

In [ ]:
df[df["Collection date"] > df["Submission date"]].shape

Find average time between collection of sample and submission of sequence collected on or after Jan 2020

In [ ]:
print(f"Number of sequences before 2020: {len(df[df["Collection date"] < pd.to_datetime("2020-01-01")])}")
df = df[df["Collection date"] >= pd.to_datetime("2020-01-01")]
df["collection_to_submission_days"] = (df["Submission date"] - df["Collection date"]).dt.days
df["collection_to_submission_days"].describe()

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df["collection_to_submission_days"], bins=50, kde=True, color='blue', stat='density')
sns.lineplot(x=sorted(df["collection_to_submission_days"]), 
             y=np.full(df.shape[0], 0), color='black', linewidth=2)
mean_days = df["collection_to_submission_days"].mean()
plt.axvline(mean_days, color='red', linestyle='--', label=f"Mean = {mean_days:.2f} days")

# Place annotation above the mean line, at the top of the plot
ymax = plt.gca().get_ylim()[1]
plt.annotate(f"Mean = {mean_days:.2f}", xy=(mean_days, ymax*0.7), xytext=(mean_days+0.5, ymax*0.85),
             arrowprops=dict(facecolor='red', arrowstyle='->'), color='red', fontsize=11)

plt.xlabel("Collection to Submission Days (days)")
plt.title(f"Distribution of time between Collection and Submission Dates | {YEAR}")
plt.legend()
plt.tight_layout()
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(PLOTS_PATH, f"collection_to_submission_days_distribution_{YEAR}.png"), dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot histograms for submission dates
sns.histplot(df["Submission date"], bins=10, color="orange", label="Submission date", kde=False, alpha=0.4)

# Calculate means
mean_submission = df["Submission date"].mean()

# Plot mean lines
plt.axvline(mean_submission, color="red", linestyle="--", label=f"Mean Submission: {mean_submission.date()}")

# Annotate the bars
ax = plt.gca()
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=10)

plt.xlabel("Date")
plt.ylabel("Count")
plt.title(f"Distribution of Submission Dates | {YEAR}")
plt.legend()
plt.tight_layout()
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(PLOTS_PATH, f"submission_dates_spread_{YEAR}.png"), dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot histograms for collection dates
sns.histplot(df["Collection date"], bins=10, color="orange", label="Collection date", kde=False, alpha=0.4)

# Calculate means
mean_collection = df["Collection date"].mean()

# Plot mean lines
plt.axvline(mean_collection, color="red", linestyle="--", label=f"Mean Collection: {mean_collection.date()}")

# Annotate the bars
ax = plt.gca()
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=10)

plt.xlabel("Date")
plt.ylabel("Count")
plt.title(f"Distribution of Collection Dates | {YEAR}")
plt.legend()
plt.tight_layout()
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(PLOTS_PATH, f"collection_dates_spread_{YEAR}.png"), dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot histograms for collection and submission dates
sns.histplot(df["Collection date"], bins=10, color="skyblue", label="Collection date", kde=False)
sns.histplot(df["Submission date"], bins=10, color="orange", label="Submission date", kde=False, alpha=0.2)

# Calculate means
mean_collection = df["Collection date"].mean()
mean_submission = df["Submission date"].mean()

# Plot mean lines
plt.axvline(mean_collection, color="blue", linestyle="--", label=f"Mean Collection: {mean_collection.date()}")
plt.axvline(mean_submission, color="red", linestyle="--", label=f"Mean Submission: {mean_submission.date()}")

# Annotate lead time
lead_time = (mean_submission - mean_collection).days
plt.annotate(
    f"Delay in submission: {lead_time} days",
    xy=(mean_submission, plt.ylim()[1]*0.8),
    xytext=(mean_submission, plt.ylim()[1]*0.95),
    arrowprops=dict(facecolor='green', edgecolor='green', arrowstyle='->', lw=2),
    color="black",
    fontsize=14,
    ha='left'
)

plt.xlabel("Date")
plt.ylabel("Count")
plt.title(f"Distribution of Collection and Submission Dates | {YEAR}")
plt.legend()
plt.tight_layout()
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(PLOTS_PATH, f"collection_submission_dates_distribution_{YEAR}.png"), dpi=300)
plt.show()

In [ ]:
df["Gender"].value_counts()

In [ ]:
df["Patient status"].value_counts()

In [ ]:
df["Patient age"].value_counts()

In [ ]:
df[~df["Gender"].isin(["Male", "Female", "unknown"])]

In [ ]:
# Swap the two values in these rows (Patient age and Patent gender)
genders = ["male", "female", "unknown"]
for i, row in df[~df["Gender"].str.lower().isin(genders)].iterrows():
    age = row["Patient age"]
    if age.lower() in genders:
        df.at[i, "Patient age"] = row["Gender"]
        df.at[i, "Gender"] = age

In [ ]:
df["Gender"].value_counts()

In [ ]:
index = 